### Introduction to Qewton and PINNs


This notebook contains a template for the joint exercise regarding physics informed deep learning and Qewton.

We start by solving the Laplace equation:

\begin{align*}
    -\Delta u &= -1 &&\text{ in } \Omega \\
    u &= 0 &&\text{ on } \partial \Omega
\end{align*}

with $\Omega = (0, 1) \times (0, 1)$.

In [16]:
# # This block is for GPU selection. Please execute.
# import pathlib
# import os
# user = int(str(pathlib.Path().resolve())[22:24])
# os.environ["CUDA_VISIBLE_DEVICES"]= str(user % 4)

Next we define the *Variables* that appear in the problem:

In [ ]:
import qewton

path = "/localdata/tomfre/SolutionData/LaplaceData/"

X = qewton.Variable("x", dim=2)
U = qewton.Variable("u", dim=1)

Now we define our domain, the unit square:

In [18]:
omega = qewton.geometries.Rectangle(X, [0.0, 0.0], 1.0, 1.0)

For the training we need to create some points, this is handle by the *Sampler*:

In [19]:
inner_sampler = qewton.RandomUniformSampler(omega, n_points=10000)
boundary_sampler = qewton.GridSampler(omega.boundary, n_points=2500)

In [20]:
# inner_points = inner_sampler.visualize().plots[0]
# boundary_points = boundary_sampler.visualize().plots[0]

# fig = qewton.viz.Figure(qewton.viz.Overlay(inner_points, boundary_points))
# fig.show()

We need a neural network that should learn our solution:

In [21]:
model = qewton.FCN(
    in_neurons=X,
    hidden_neurons=20,
    out_neurons=U,
    n_hidden_layers=3,
    activation=qewton.bb.Tanh,
    )

Now, we have to transform out mathematical conditions given by our PDE into corresponding training conditions. First for the differential equation itself:

In [22]:
def pde_residual(u : U, x : X):
    return -u.laplacian(x) + 1.0

pde_graph = qewton.PINNPipeline(
    sampler=inner_sampler,
    residual=pde_residual,
    models=[model]
)
pde_constraint = pde_graph.constraint

In [23]:
# pde_graph.sort()
# qewton.viz.Figure(pde_graph.diagram(depth=2)).show()

Next for the boundary condition:

In [ ]:
def boundary_residual(u : U):
    return u

boundary_constraint = qewton.constraints.PINNConstraint(
    boundary_residual, name="BoundaryConstraint", weight=50.0
)
boundary_graph = qewton.PINNPipeline(boundary_sampler, [model], boundary_constraint)

For training, we collect all conditions and choose our optimizer:

In [29]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=5000,
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=adam_phase,
    graphs=[pde_graph, boundary_graph],
    training_objectives=[pde_constraint, boundary_constraint],
    device=qewton.cuda(0),
)

trainer.run()

Optimization Phase 1:   0%|          | 12/5000 [00:00<01:04, 77.51it/s, loss=0.16]  

Optimization Phase 1: 100%|██████████| 5000/5000 [00:58<00:00, 85.71it/s, loss=0.00131]


Now we compare with a reference solution:

In [30]:
model.to(qewton.cpu)
coords_grid = qewton.DEFAULT_DL_BACKEND.load_data(path + "coordinates.pt")
fem_grid = qewton.DEFAULT_DL_BACKEND.load_data(path + "solution.pt")

data_config = qewton.DataConfiguration(
    qewton.GeometryAxes(qewton.geometries.GridGeometry(X, coords_grid)),
    qewton.FeatureAxes(variable=U)
)

layout = pde_graph.visualize(model.output_ports[0], 
                             reference=fem_grid, 
                             reference_config=data_config)
qewton.viz.Figure(layout).show()